# Exciton transport

Standalone run on the supplied **SWG 256×256 geometry, 12 µm × 12 µm**. Edit the **Parameters** cell, then select **Kernel → Restart Kernel and Run All Cells**. Paths are relative to the repository root. Results are saved under `outputs/`.

The default input is the validated FvK SWG strain map bundled with the repository, so this notebook runs without another notebook or model weights. To use your own mechanics, first run `03_fvk.ipynb`, then set `strain_file = "outputs/swg_fvk.npz"` and `strain_key = "strain_fraction"`. For your device, also select the corresponding support map and aligned optical pump.

To change geometry, material, loading or physical size, recompute mechanics and use that result instead of the fixed reference strain.


In [ ]:
%matplotlib inline
from pathlib import Path
import os
import sys

root_candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in root_candidates if (p / "plate_models.py").is_file()
             and (p / "data/swg_256.npz").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or its notebooks directory")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository:", ROOT)

from pathlib import Path
import numpy as np
from exciton_solver_fixed import solve_exciton_fixed


## Parameters

Change the input file/key, physical domain size, physical parameters and output path here. Support maps use 1 for supports and 0 for suspended film. `width_um` sets **both** length and width of the square domain; independent unequal x/y extents are not exposed by these solvers. Changing the domain size scales every feature; change the input pixels to change individual teeth or gaps.
Transport units: D in m²/s, lifetime in s, mobility in m²/(eV·s), alpha in eV per **percent** strain. Input strain is **dimensionless**. `width_um` is used for reference files without stored width; calculated files supply their saved `width_m`. Pump amplitude and strain must share physical domain, orientation and grid.


In [ ]:
structure_file = "data/swg_256.npz"
structure_key = "array"
strain_file = "data/reference/swg_fvk_reference.npz"
strain_key = "strain_A"
width_um = 12.0
pump_file = "data/reference/pump_532nm_w2.txt"
generation_peak = 1e21
temperature_k = 5.0
diffusion = 2.4e-4
lifetime_s = 498e-12
mobility = diffusion / (8.617e-5 * temperature_k)
alpha = -0.0315
waveguide_center_um = 6.0
waveguide_half_width_um = 0.5
output_file = "outputs/swg_excitons.npz"


## Calculate and save

At these SWG parameters the waveguide peak is approximately 2.62907226e12 under the given generation normalization. `waveguide_center_um` and `waveguide_half_width_um` define the manuscript's measurement band, centered at y=6 µm with half-width 0.5 µm. Change that measurement region for your geometry; it does not affect the transport solve.

The solver returns a density array with zero outer-boundary density. G must be nonnegative; density has the units of G multiplied by seconds (G in m⁻²s⁻¹ gives density in m⁻²). D and tau must be positive. The example computes mobility from the Einstein relation; set `mobility` directly to use another measured value. `alpha` is in eV per **percent** strain, while input strain is **dimensionless**: the solver converts it internally. Divide dataset or U-Net strain in percent by 100 before passing it to this solver.

The explicit transport values here select the manuscript's 5 K case. Omitting them selects the historical defaults D=0.97e-4, tau=1.02e-9, mu=0.0169, alpha=−0.0632. For a new grid, resample and align your optical field to that grid before solving. For an already normalized generation-rate map, pass it directly as `generation` without squaring it again.

In [ ]:
with np.load(strain_file) as data:
    strain_fraction = data[strain_key]
    width_m = float(data["width_m"]) if "width_m" in data else width_um * 1e-6
with np.load(structure_file) as data:
    support = data[structure_key]
if support.shape != strain_fraction.shape:
    raise ValueError("Support and strain maps must have the same grid")
pump_amplitude = np.loadtxt(pump_file)
if pump_amplitude.shape != strain_fraction.shape:
    raise ValueError("Pump and strain maps must have the same grid")
pump_intensity = np.abs(pump_amplitude)**2
generation = generation_peak * pump_intensity / pump_intensity.max()
density = solve_exciton_fixed(strain_fraction, width_m, generation,
                              D=diffusion, tau=lifetime_s, mu=mobility, alpha=alpha)
Path(output_file).parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(output_file, density=density, generation=generation,
                    strain_fraction=strain_fraction, width_m=width_m,
                    D=diffusion, tau=lifetime_s, mu=mobility, alpha=alpha)
coords_um = np.linspace(0, width_m * 1e6, density.shape[0])
waveguide = np.broadcast_to(
    (np.abs(coords_um - waveguide_center_um) < waveguide_half_width_um)[:, None], density.shape
)
print("Global density peak:", density.max())
print("SWG waveguide density peak:", density[waveguide].max())


## Inspect the result

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
im = ax.imshow(density, origin="lower", extent=[0, width_m * 1e6, 0, width_m * 1e6])
ax.set(title="Exciton density", xlabel="x (um)", ylabel="y (um)")
fig.colorbar(im, ax=ax)
plt.show()
